# LeetCode 3266 — Final Array State After K Multiplication Operations II bằng Python

- Link: https://leetcode.com/problems/final-array-state-after-k-multiplication-operations-ii/
- Độ khó: **Hard**
- Chủ đề: **Mô phỏng bằng heap tới khi trạng thái "ổn định"**, sau đó các phép nhân đi theo **chu kỳ** `n` phép ⇒ dùng `divmod(k, n)` và **lũy thừa nhanh modulo**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu phép toán: mỗi lần lấy phần tử **nhỏ nhất** (hòa thì chỉ số nhỏ nhất) nhân với `multiplier`; `k` tới `10^9`, chỉ lấy modulo **ở cuối**.
2. Pha 1 — mô phỏng bằng heap `(giá trị, chỉ số)` khi `min · multiplier ≤ max ban đầu`: mỗi phần tử chỉ bị nhân `O(log_m(max))` lần ⇒ tổng `O(n log)` bước.
3. Pha 2 — khi `min · multiplier > max`, mọi phần tử nằm trong một "dải" hệ số `m`: các phép tiếp theo nhân lần lượt từng phần tử theo thứ tự `(giá trị, chỉ số)`, lặp lại theo chu kỳ `n`.
4. Phần còn lại `k = q·n + r`: mọi phần tử nhân `m^q`, `r` phần tử đầu theo thứ tự nhân thêm `m`.
5. Trường hợp đặc biệt `multiplier = 1`.

## 2. Đề bài & ví dụ

Thực hiện `k` phép trên `nums`: tìm giá trị nhỏ nhất `x` (nếu nhiều, chọn phần tử xuất hiện **đầu tiên**), thay bằng `x · multiplier`. Sau `k` phép, lấy mỗi phần tử modulo `10^9 + 7`. Trả về mảng cuối.

**Ví dụ 1:** `nums = [2,1,3,5,6]`, `k = 5`, `multiplier = 2` → `[8,4,6,5,6]`.
**Ví dụ 2:** `nums = [100000,2000]`, `k = 2`, `multiplier = 1000000` → `[999999307,999999993]`.

**Ràng buộc:** `1 ≤ n ≤ 10^4`; `1 ≤ nums[i] ≤ 10^9`; `1 ≤ k ≤ 10^9`; `1 ≤ multiplier ≤ 10^6`.

In [1]:
from typing import List
import heapq
MOD = 10 ** 9 + 7

examples = [([2, 1, 3, 5, 6], 5, 2, [8, 4, 6, 5, 6]), ([100000, 2000], 2, 1000000, [999999307, 999999993])]
for nums, k, m, exp in examples:
    print(nums, 'k =', k, 'm =', m, '-> kỳ vọng', exp)

[2, 1, 3, 5, 6] k = 5 m = 2 -> kỳ vọng [8, 4, 6, 5, 6]
[100000, 2000] k = 2 m = 1000000 -> kỳ vọng [999999307, 999999993]


## 3. Kiến thức nền

### 3.1. Mô phỏng trực tiếp (số nguyên lớn của Python)

In [2]:
def simulate(nums, k, m):
    a = list(nums)
    for _ in range(k):
        i = min(range(len(a)), key=lambda t: (a[t], t))
        a[i] *= m
    return a

print(simulate([2, 1, 3, 5, 6], 5, 2), '(chưa lấy modulo)')

[8, 4, 6, 5, 6] (chưa lấy modulo)


### 3.2. Vì sao có chu kỳ?

Khi `min · m > max`, phần tử nhỏ nhất sau khi nhân trở thành **lớn nhất** (lớn hơn mọi phần tử khác). Do đó các phép tiếp theo đi lần lượt qua các phần tử theo thứ tự tăng `(giá trị, chỉ số)`; sau `n` phép mọi phần tử nhân `m` một lần, thứ tự tương đối giữ nguyên ⇒ lặp lại.

In [3]:
a = [5, 7, 9]           # 5·2 = 10 > 9: đã ổn định
print([simulate(a, t, 2) for t in range(7)])

[[5, 7, 9], [10, 7, 9], [10, 14, 9], [10, 14, 18], [20, 14, 18], [20, 28, 18], [20, 28, 36]]


### 3.3. Pha 1 ngắn

Mỗi lần phần tử `x` bị nhân trong pha 1 thì `x · m ≤ max₀`, nên một phần tử bị nhân tối đa `log_m(max₀)` lần (≤ 30 với `m ≥ 2`). Tổng số bước pha 1 `≤ 30 n`. Với `m = 1`, mọi phép không đổi gì ⇒ trả mảng gốc (mod).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`m = 2`, `max₀ = 6`): pha 1 chạy khi `min · 2 ≤ 6`:

- `1 → 2`, `2 (chỉ số 0) → 4`, `2 → 4`, `3 → 6` — sau 4 phép: `[4, 4, 6, 5, 6]`, `min = 4`, `4·2 = 8 > 6` ⇒ dừng.
- Còn `k = 1`: `q = 0`, `r = 1` ⇒ phần tử đầu theo thứ tự `(4, 0)` nhân 2 ⇒ `[8, 4, 6, 5, 6]` ✓.

## 5. Cách 1 — Brute Force: mô phỏng `k` phép

In [4]:
def brute(nums, k, m):
    return [x % MOD for x in simulate(nums, k, m)]

for nums, k, m, exp in examples:
    print(brute(nums, k, m), '| kỳ vọng', exp)

[8, 4, 6, 5, 6] | kỳ vọng [8, 4, 6, 5, 6]
[999999307, 999999993] | kỳ vọng [999999307, 999999993]


**Độ phức tạp:** `O(k · n)` và số nguyên khổng lồ — chỉ cho `k` nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
if m == 1: return nums mod p
mx = max(nums); h = heap của (x, i)
while k > 0 and h[0].x * m <= mx: pop (x, i), push (x*m, i), k -= 1
q, r = divmod(k, n)
thứ tự = sorted(h)                       # theo (giá trị, chỉ số)
mỗi phần tử: x * m^q (mod p); r phần tử đầu theo thứ tự: nhân thêm m
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Heap mô phỏng đủ `k` phép (`O(k log n)`)

In [5]:
def heap_full(nums, k, m):
    h = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(h)
    for _ in range(k):
        x, i = heapq.heappop(h)
        heapq.heappush(h, (x * m, i))
    res = [0] * len(nums)
    for x, i in h:
        res[i] = x % MOD
    return res

for nums, k, m, exp in examples:
    print(heap_full(nums, k, m), '| kỳ vọng', exp)

[8, 4, 6, 5, 6] | kỳ vọng [8, 4, 6, 5, 6]
[999999307, 999999993] | kỳ vọng [999999307, 999999993]


Đúng nhưng `k = 10^9` là quá lớn, và giá trị không lấy modulo được trong lúc mô phỏng (vì cần so sánh).

### 7.2. Cách 3 — Heap tới khi ổn định + chu kỳ (lời giải chính)

In [6]:
def heap_then_cycle(nums, k, m):
    n = len(nums)
    if m == 1:
        return [x % MOD for x in nums]
    mx = max(nums)
    h = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(h)
    while k and h[0][0] * m <= mx:
        x, i = heapq.heappop(h)
        heapq.heappush(h, (x * m, i))
        k -= 1
    q, r = divmod(k, n)
    order = sorted(h)
    res = [0] * n
    base = pow(m, q, MOD)
    for idx, (x, i) in enumerate(order):
        v = x % MOD * base % MOD
        if idx < r:
            v = v * m % MOD
        res[i] = v
    return res

for nums, k, m, exp in examples:
    print(heap_then_cycle(nums, k, m), '| kỳ vọng', exp)

[8, 4, 6, 5, 6] | kỳ vọng [8, 4, 6, 5, 6]
[999999307, 999999993] | kỳ vọng [999999307, 999999993]


### 7.3. Cách 4 — Pha 1 bằng sắp xếp theo "số lần nhân" (không heap)

Với mỗi phần tử `x`, gọi `c(x)` = số lần nhỏ nhất để `x · m^{c} · m > mx`, tức số lần nó bị nhân trong pha 1 nếu `k` đủ lớn. Nếu `Σ c(x) ≤ k`, pha 1 kết thúc đúng sau `Σ c` phép (mọi phần tử đã nhân `c(x)` lần, thứ tự không quan trọng vì kết quả là cố định). Nếu `Σ c > k`, quay về heap (hoặc tìm kiếm nhị phân theo ngưỡng). Ở đây ta dùng nó để tăng tốc khi `k` lớn.

In [7]:
def counted_phase(nums, k, m):
    n = len(nums)
    if m == 1:
        return [x % MOD for x in nums]
    mx = max(nums)
    cs = []
    for x in nums:
        c = 0
        while x * m <= mx:
            x *= m; c += 1
        cs.append((x, c))
    total = sum(c for _, c in cs)
    if total > k:
        return heap_then_cycle(nums, k, m)
    k -= total
    q, r = divmod(k, n)
    order = sorted((x, i) for i, (x, _) in enumerate(cs))
    res = [0] * n
    base = pow(m, q, MOD)
    for idx, (x, i) in enumerate(order):
        v = x % MOD * base % MOD
        res[i] = v * m % MOD if idx < r else v
    return res

for nums, k, m, exp in examples:
    print(counted_phase(nums, k, m), '| kỳ vọng', exp)

[8, 4, 6, 5, 6] | kỳ vọng [8, 4, 6, 5, 6]
[999999307, 999999993] | kỳ vọng [999999307, 999999993]


Điểm tinh tế: nếu `k ≥ Σ c(x)`, heap ở cách 3 cũng sẽ nhân **đúng** `c(x)` lần mỗi phần tử trong pha 1 (phần tử chỉ bị chọn khi nó là min và `min·m ≤ mx`), nên trạng thái sau pha 1 giống nhau.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 với `k = 12`.

In [8]:
def trace(nums, k, m):
    mx = max(nums); h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h)
    step = 0
    while k and h[0][0] * m <= mx:
        x, i = heapq.heappop(h); heapq.heappush(h, (x * m, i)); k -= 1; step += 1
        print(f'  pha 1, bước {step}: nhân chỉ số {i} ({x} -> {x * m})')
    q, r = divmod(k, len(nums))
    print('  sau pha 1:', sorted(h), '| còn k =', k, '-> q =', q, ', r =', r)
    print('đáp án:', heap_then_cycle(nums, step + k, m), '| mô phỏng:', brute(nums, step + k, m))

trace([2, 1, 3, 5, 6], 12, 2)

  pha 1, bước 1: nhân chỉ số 1 (1 -> 2)
  pha 1, bước 2: nhân chỉ số 0 (2 -> 4)
  pha 1, bước 3: nhân chỉ số 1 (2 -> 4)
  pha 1, bước 4: nhân chỉ số 2 (3 -> 6)
  sau pha 1: [(4, 0), (4, 1), (5, 3), (6, 2), (6, 4)] | còn k = 8 -> q = 1 , r = 3
đáp án: [16, 16, 12, 20, 12] | mô phỏng: [16, 16, 12, 20, 12]


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        n = len(nums)
        if multiplier == 1:
            return [x % MOD for x in nums]
        mx = max(nums)
        h = [(x, i) for i, x in enumerate(nums)]
        heapq.heapify(h)
        while k and h[0][0] * multiplier <= mx:
            x, i = h[0]
            heapq.heapreplace(h, (x * multiplier, i))
            k -= 1
        q, r = divmod(k, n)
        h.sort()
        base = pow(multiplier, q, MOD)
        res = [0] * n
        for idx, (x, i) in enumerate(h):
            v = x % MOD * base % MOD
            if idx < r:
                v = v * multiplier % MOD
            res[i] = v
        return res

for nums, k, m, exp in examples:
    got = Solution().getFinalState(nums, k, m)
    print(got, 'OK' if got == exp else 'SAI')

[8, 4, 6, 5, 6] OK
[999999307, 999999993] OK


## 10. Vì sao thuật toán đúng

1. **Pha 1** mô phỏng đúng các phép (heap theo `(giá trị, chỉ số)` chọn đúng phần tử đề yêu cầu).
2. **Bất biến pha 1:** mọi giá trị luôn `≤ mx` (phần tử chỉ bị nhân khi `x·m ≤ mx`). Khi dừng (`min·m > mx ≥ max`), ta có `max < min·m`.
3. **Chu kỳ:** gọi thứ tự hiện tại `e_1 < e_2 < … < e_n` (theo `(giá trị, chỉ số)`). Phép tiếp theo nhân `e_1`, và `e_1·m > e_n` ⇒ `e_1·m` trở thành lớn nhất; dải mới `e_2 < … < e_n < e_1·m` vẫn thỏa `max < min·m` (`e_1·m < e_2·m`). Quy nạp: các phép đi theo vòng `e_1, e_2, …, e_n, e_1, …`. (Hòa giá trị giữa `e_1·m` và phần tử khác không thể xảy ra vì `e_1·m > max`.)
4. **Modulo cuối:** `x · m^{q} · m^{[idx < r]}` — phép nhân tương thích modulo, chỉ lấy modulo sau khi đã quyết định thứ tự.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(k · n)` | `O(n)` |
| 2. Heap đủ `k` phép | `O(k log n)` | `O(n)` |
| 3. Heap + chu kỳ | `O(n log n · log_m(max) + log q)` | `O(n)` |
| 4. Đếm số lần nhân | `O(n log_m(max) + n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'heap_full': heap_full, 'heap_then_cycle': heap_then_cycle, 'counted_phase': counted_phase,
         'submit': lambda a, k, m: Solution().getFinalState(a, k, m)}
edge = [([5], 3, 2, [40]), ([1, 1, 1], 4, 3, [9, 3, 3]), ([3, 3], 1, 1, [3, 3]), ([1, 2], 1, 2, [2, 2]),
        ([1, 2], 2, 2, [4, 2]), ([7, 1], 5, 10, None)]
for nums, k, m, exp in examples + edge:
    e = brute(nums, k, m)
    if exp is not None:
        assert e == exp, (nums, k, m, e)
    for name, f in impls.items():
        assert f(nums, k, m) == e, (name, nums, k, m)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [11]:
import random
rng = random.Random(3266)
cnt = 0
for _ in range(800):
    n = rng.randint(1, 6)
    nums = [rng.randint(1, rng.choice([5, 50, 1000])) for _ in range(n)]
    k = rng.randint(1, 40)
    m = rng.choice([1, 2, 3, 7, 10])
    exp = brute(nums, k, m)
    for name, f in impls.items():
        assert f(nums, k, m) == exp, (name, nums, k, m)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test (kể cả multiplier = 1).')

Khớp mô phỏng trên 800 bộ test (kể cả multiplier = 1).


### 12.3. Ca lớn `n = 10^4`, `k = 10^9`

In [12]:
import time
N = 10 ** 4
big = {
    'ngẫu nhiên, m = 2': ([rng.randint(1, 10 ** 9) for _ in range(N)], 10 ** 9, 2),
    'm = 10^6': ([rng.randint(1, 10 ** 9) for _ in range(N)], 10 ** 9, 10 ** 6),
    'toàn 1, m = 3': ([1] * N, 10 ** 9, 3),
    'm = 1': ([rng.randint(1, 10 ** 9) for _ in range(N)], 10 ** 9, 1),
}
for name, (nums, k, m) in big.items():
    t = time.time(); a = Solution().getFinalState(nums, k, m); t1 = time.time() - t
    t = time.time(); b = counted_phase(nums, k, m); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: nộp {t1:.2f}s | đếm số lần nhân {t2:.2f}s | 3 phần tử đầu {a[:3]}")
q, r = divmod(10 ** 9, N)
assert Solution().getFinalState([1] * N, 10 ** 9, 3) == [pow(3, q + (i < r), MOD) for i in range(N)]
print('toàn 1: phần tử i nhận 3^(q + [i < r]) — đúng.')

ngẫu nhiên, m = 2   : nộp 0.01s | đếm số lần nhân 0.01s | 3 phần tử đầu [131283369, 497172852, 615560549]
m = 10^6            : nộp 0.00s | đếm số lần nhân 0.01s | 3 phần tử đầu [469808612, 757521475, 363624843]
toàn 1, m = 3       : nộp 0.00s | đếm số lần nhân 0.00s | 3 phần tử đầu [916902199, 916902199, 916902199]
m = 1               : nộp 0.00s | đếm số lần nhân 0.00s | 3 phần tử đầu [643062695, 796363138, 46844943]
toàn 1: phần tử i nhận 3^(q + [i < r]) — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy modulo **trong khi** mô phỏng rồi so sánh giá trị đã modulo

In [13]:
def wrong_mod_early(nums, k, m):
    h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h)
    for _ in range(k):
        x, i = heapq.heappop(h)
        heapq.heappush(h, (x * m % MOD, i))                       # SAI: thứ tự bị phá
    res = [0] * len(nums)
    for x, i in h:
        res[i] = x
    return res

bad = ([10 ** 9, 10 ** 9 - 1], 3, 2)
print('sai :', wrong_mod_early(*bad), '| đúng:', Solution().getFinalState(*bad))

sai : [1000000000, 999999943] | đúng: [999999993, 999999975]


**Vì sao sai:** `(10^9 − 1)·2 mod p` là số **nhỏ** dù giá trị thật rất lớn — phép tiếp theo chọn nhầm phần tử. Đề nói rõ chỉ lấy modulo **sau** `k` phép.

### Lỗi 2 — Hòa giá trị thì chọn chỉ số **lớn** hơn

In [14]:
def wrong_tie(nums, k, m):
    h = [(x, -i) for i, x in enumerate(nums)]; heapq.heapify(h)          # SAI
    for _ in range(k):
        x, ni = heapq.heappop(h); heapq.heappush(h, (x * m, ni))
    res = [0] * len(nums)
    for x, ni in h:
        res[-ni] = x % MOD
    return res

print('sai :', wrong_tie(*examples[0][:3]), '| đúng:', Solution().getFinalState(*examples[0][:3]))

sai : [4, 8, 6, 5, 6] | đúng: [8, 4, 6, 5, 6]


**Vì sao sai:** đề chọn phần tử nhỏ nhất **xuất hiện đầu tiên**.

### Lỗi 3 — Pha 2 chia phần dư `r` theo **chỉ số** thay vì theo thứ tự `(giá trị, chỉ số)`

In [15]:
def wrong_remainder_by_index(nums, k, m):
    n = len(nums)
    if m == 1:
        return [x % MOD for x in nums]
    mx = max(nums); h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h)
    while k and h[0][0] * m <= mx:
        x, i = heapq.heappop(h); heapq.heappush(h, (x * m, i)); k -= 1
    q, r = divmod(k, n)
    res = [0] * n
    for x, i in h:
        res[i] = x % MOD * pow(m, q, MOD) % MOD * (m if i < r else 1) % MOD      # SAI
    return res

bad = ([5, 3], 3, 2)
print('sai :', wrong_remainder_by_index(*bad), '| đúng:', Solution().getFinalState(*bad))

sai : [20, 6] | đúng: [10, 12]


**Vì sao sai:** trong chu kỳ, phần tử bị nhân trước là phần tử **nhỏ nhất**, không phải chỉ số nhỏ nhất.

### Lỗi 4 — Quên trường hợp `multiplier = 1` (vòng pha 1 chạy `k` lần)

In [16]:
def wrong_no_m1(nums, k, m, guard=10 ** 6):
    mx = max(nums); h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h)
    steps = 0
    while k and h[0][0] * m <= mx:                                # với m = 1: luôn đúng
        x, i = heapq.heappop(h); heapq.heappush(h, (x * m, i)); k -= 1; steps += 1
        if steps >= guard:
            return f'dừng ép sau {guard} bước (còn k = {k}) — sẽ chạy tới 1e9 bước'
    return 'xong'

print(wrong_no_m1([5, 7], 10 ** 9, 1))

dừng ép sau 1000000 bước (còn k = 999000000) — sẽ chạy tới 1e9 bước


**Vì sao sai:** với `m = 1`, `min · 1 ≤ max` luôn đúng ⇒ pha 1 không bao giờ "ổn định", vòng lặp chạy đủ `10^9` lần (quá thời gian). Phải xử lý riêng: `m = 1` không làm thay đổi mảng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `k` khổng lồ với phép lặp ⇒ tìm **pha chuyển tiếp ngắn** rồi **chu kỳ**.
- Phép nhân đẩy phần tử nhỏ thành lớn ⇒ sau một lúc mọi phần tử "xoay vòng".
- Modulo chỉ an toàn khi không cần so sánh giá trị nữa.

## 15. Biến thể của bài

1. **LC 3264 (bản I):** `k ≤ 10`, mô phỏng trực tiếp.
2. Chọn phần tử **lớn nhất** để chia (làm tròn xuống) — không có chu kỳ đẹp (hội tụ về 0/1).
3. Phép cộng `x + d` thay vì nhân ⇒ chu kỳ khi các phần tử nằm trong dải độ rộng `d`.
4. Trả về tổng các phần tử sau `k` phép.

In [17]:
# Biến thể 3: cộng d
def add_variant(nums, k, d):
    n = len(nums); mx = max(nums)
    h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h)
    while k and h[0][0] + d <= mx:
        x, i = heapq.heappop(h); heapq.heappush(h, (x + d, i)); k -= 1
    q, r = divmod(k, n)
    order = sorted(h); res = [0] * n
    for idx, (x, i) in enumerate(order):
        res[i] = x + d * q + (d if idx < r else 0)
    return res

def add_brute(nums, k, d):
    a = list(nums)
    for _ in range(k):
        i = min(range(len(a)), key=lambda t: (a[t], t)); a[i] += d
    return a

rng = random.Random(15)
for _ in range(400):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 6))]
    k = rng.randint(1, 50); d = rng.randint(1, 7)
    assert add_variant(nums, k, d) == add_brute(nums, k, d)
print('Biến thể phép cộng d: cùng khung "ổn định rồi xoay vòng" — khớp mô phỏng.')

Biến thể phép cộng d: cùng khung "ổn định rồi xoay vòng" — khớp mô phỏng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng | `O(kn)` | Oracle |
| 2 | Heap đủ `k` phép | `O(k log n)` | `k` nhỏ |
| 3 | Heap tới ổn định + chu kỳ | `O(n log n log_m M)` | **Nộp bài** |
| 4 | Đếm số lần nhân pha 1 | `O(n log_m M)` | Tránh heap khi `k` lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1]`, `k = 10`, `m = 2`.

**Bài 2 (tính tay):** `nums = [3, 1]`, `k = 3`, `m = 2`.

**Bài 3 (điền TODO):** số phép của **pha 1** (trước khi ổn định), không quá `k`.

**Bài 4 (tự viết lại):** biến thể 4 — tổng các phần tử (mod p) sau `k` phép.

**Bài 5 (mini challenge):** chỉ số phần tử được nhân ở phép thứ `t` (`t ≤ 10^18`).

In [18]:
def phase1_length_todo(nums, k, m):
    # TODO
    raise NotImplementedError

def final_sum_todo(nums, k, m):
    # TODO
    raise NotImplementedError

def index_at_step_todo(nums, m, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một phần tử. (2) `2^10`. (3) `[1024]`.

**Bài 2.** (1) `1→2`, `2→4`, `3→6`? Thứ tự: `[3,1] → [3,2] → [3,4] → [6,4]`. (2) Kiểm từng bước. (3) `[6, 4]`.

**Bài 3.** (1) Đếm bước trong vòng `while`. (2) Hoặc `min(k, Σ c(x))`. (3) Cẩn thận `m = 1`.

**Bài 4.** (1) Tổng các `res[i]`. (2) Modulo. (3) Đối chiếu mô phỏng.

**Bài 5.** (1) Nếu `t ≤` độ dài pha 1: mô phỏng. (2) Ngược lại: `(t − L − 1) mod n` cho vị trí trong thứ tự ổn định. (3) Trả chỉ số tương ứng.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3, 4
assert Solution().getFinalState([1], 10, 2) == [1024]
assert Solution().getFinalState([3, 1], 3, 2) == [6, 4] == brute([3, 1], 3, 2)

def phase1_length(nums, k, m):
    if m == 1:
        return k
    mx = max(nums); h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h); L = 0
    while L < k and h[0][0] * m <= mx:
        x, i = heapq.heappop(h); heapq.heappush(h, (x * m, i)); L += 1
    return L

def final_sum(nums, k, m):
    return sum(Solution().getFinalState(nums, k, m)) % MOD

assert phase1_length([2, 1, 3, 5, 6], 100, 2) == 4
for _ in range(200):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 5))]; k = rng.randint(1, 30); m = rng.choice([2, 3, 5])
    assert final_sum(nums, k, m) == sum(brute(nums, k, m)) % MOD
print('Bài 1: [1024] | Bài 2: [6, 4] | Bài 3: pha 1 của ví dụ 1 dài 4 | Bài 4 khớp mô phỏng.')

Bài 1: [1024] | Bài 2: [6, 4] | Bài 3: pha 1 của ví dụ 1 dài 4 | Bài 4 khớp mô phỏng.


In [20]:
# Bài 5
def index_at_step(nums, m, t):
    """Chỉ số được nhân ở phép thứ t (t >= 1)."""
    mx = max(nums); h = [(x, i) for i, x in enumerate(nums)]; heapq.heapify(h); L = 0
    while h[0][0] * m <= mx:
        x, i = heapq.heappop(h); L += 1
        if L == t:
            return i
        heapq.heappush(h, (x * m, i))
    order = sorted(h)
    return order[(t - L - 1) % len(nums)][1]

def index_brute(nums, m, t):
    a = list(nums)
    for _ in range(t - 1):
        i = min(range(len(a)), key=lambda s: (a[s], s)); a[i] *= m
    return min(range(len(a)), key=lambda s: (a[s], s))

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 60) for _ in range(rng.randint(1, 5))]; m = rng.choice([2, 3, 4]); t = rng.randint(1, 40)
    assert index_at_step(nums, m, t) == index_brute(nums, m, t)
print('Bài 5 khớp mô phỏng; ví dụ phép thứ 10^18 của ví dụ 1:', index_at_step([2, 1, 3, 5, 6], 2, 10 ** 18))

Bài 5 khớp mô phỏng; ví dụ phép thứ 10^18 của ví dụ 1: 0


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if m == 1: return nums mod p
mx = max(nums); heap (x, i)
while k and heap[0].x * m <= mx: nhân phần tử đầu heap; k -= 1
q, r = divmod(k, n); sắp heap theo (x, i)
res[i] = x * m^q (* m nếu thuộc r phần tử đầu)  mod p
```

**Checklist:**

- [ ] Xử lý `multiplier = 1`?
- [ ] Không lấy modulo trước khi xong so sánh?
- [ ] Hòa ⇒ chỉ số nhỏ hơn trước?
- [ ] Phần dư `r` theo thứ tự `(giá trị, chỉ số)`?
- [ ] `pow(m, q, MOD)` cho `q` lớn?